# Fine-Tuning T5-Small for Smart Location Geocoding

**Paper:** *Smart Location Geocoding using Large Language Models*  
Maheshwari · Vasudevan · Ramudu · Panigrahi (2025)  
VIT Chennai & DRDO CAIR Bangalore

---

### What this notebook does
1. Installs all required libraries
2. Mounts your Google Drive (model + dataset live there)
3. Loads and splits `final_training_geocoded.csv` with **stratified sampling by state** (as in the paper)
4. Fine-tunes a **fresh T5-small** (60M params) using HuggingFace Trainer API
5. Evaluates with **fuzzy accuracy** (SequenceMatcher ≥ 0.85) — same metric as the paper
6. Saves the finished model to Google Drive as a drop-in replacement for `t5_corrector_final/`

> ⚡ **Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU)  
> The full dataset (~500K rows) takes ~45 min on a T4. Use `MAX_TRAIN_SAMPLES` below to do a quick test run first.

In [ ]:
# ── Cell 1: Install required libraries ───────────────────────────────────────
# Colab already has torch, pandas, numpy. We only need to add:
#   transformers + datasets (HuggingFace), sentencepiece (T5 tokenizer), scikit-learn (stratified split)
!pip install -q transformers==4.40.0 datasets sentencepiece scikit-learn accelerate

print("✅ Libraries installed")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import warnings
import logging
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports done  |  Device: {device.upper()}")

## 📂 Mount Google Drive

Upload `final_training_geocoded.csv` to your Drive once, and the trained model will also be saved back there so you never lose it when the Colab session ends.

**Before running this cell**, upload `final_training_geocoded.csv` to:
```
My Drive/t5_location/final_training_geocoded.csv
```
The trained model will be saved to:
```
My Drive/t5_location/t5_corrector_final/
```

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Paths — these match the folder structure described in the markdown cell above.
# ✏️ Change DRIVE_FOLDER if you put the CSV somewhere else in your Drive.
DRIVE_FOLDER  = "/content/drive/MyDrive/t5_location"
DATA_PATH     = os.path.join(DRIVE_FOLDER, "final_training_geocoded.csv")
OUTPUT_DIR    = os.path.join(DRIVE_FOLDER, "t5_corrector_final")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Drive mounted")
print(f"   Dataset  : {DATA_PATH}")
print(f"   Model out: {OUTPUT_DIR}")

In [ ]:
# ── Cell 4: Training Configuration ───────────────────────────────────────────
# All paper hyperparameters are here. Change anything you need in one place.

MODEL_NAME   = "t5-small"    # Fresh T5-small from HuggingFace (60M params)
RANDOM_SEED  = 42

# ✏️ EPOCHS — Paper used 2. Increase for better accuracy, decrease for quick tests.
NUM_EPOCHS = 2               # Change epochs here (paper: 2)

# ✏️ BATCH SIZE — Paper used 8 (on Tesla P100 16GB). Lower this to 4 if you get OOM errors.
BATCH_SIZE = 8               # Change batch size here (paper: 8)

# ✏️ GRADIENT ACCUMULATION — Effective batch = BATCH_SIZE × GRAD_ACCUM = 16 (paper default)
GRAD_ACCUM = 2               # Change gradient accumulation steps here (paper: 2)

# ✏️ MAX_TRAIN_SAMPLES — Set to None to use the full ~500K dataset (recommended).
#    Set to a number like 50000 for a quick test run (finishes in a few minutes).
MAX_TRAIN_SAMPLES = None     # e.g. 50000 for quick test, None for full training

# ✏️ LEARNING RATE — Standard for T5-small seq2seq tasks. Lower = more stable, slower.
LEARNING_RATE = 5e-4         # Change learning rate here

WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.05       # 5% of total steps used for warmup (paper default)
FUZZY_THRESHOLD = 0.85       # Accuracy threshold from accuracy.py (paper metric)
MAX_SEQ_LEN     = 64         # Location names are short; 64 tokens is plenty
TASK_PREFIX     = "rectify location: "  # Standard T5 task prefix

# Auto-detect FP16: enabled on GPU, disabled on CPU
USE_FP16 = torch.cuda.is_available()

print("✅ Configuration:")
print(f"   Model          : {MODEL_NAME}")
print(f"   Epochs         : {NUM_EPOCHS}")
print(f"   Batch size     : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"   Learning rate  : {LEARNING_RATE}")
print(f"   FP16           : {USE_FP16}")
print(f"   Max train rows : {MAX_TRAIN_SAMPLES or 'all'}")

## 📊 Load & Split Data

Loads `final_training_geocoded.csv` and creates a **stratified train / val / test split by Indian state** — exactly as described in Section VI-B of the paper.

Split: **80% train / 10% validation / 10% test**

In [ ]:
# ── Cell 5: Load & Split Data ─────────────────────────────────────────────────
print(f"Loading dataset from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Total rows loaded : {len(df):,}")
print(f"Columns           : {df.columns.tolist()}")

# Drop rows with missing inputs, targets, or state (needed for stratification)
df = df.dropna(subset=["noisy_address", "correct_address", "state"])
df = df[df["noisy_address"].str.strip() != ""]
df = df[df["correct_address"].str.strip() != ""]
print(f"Rows after cleaning: {len(df):,}")
print(f"\nState distribution (top 10):")
print(df["state"].value_counts().head(10))

# Build input/target pairs. The noisy_address_augmented column (heavier noise)
# is also included as extra training pairs for better generalisation.
pairs = df[["noisy_address", "correct_address", "state"]].rename(
    columns={"noisy_address": "input", "correct_address": "target"}
)

if "noisy_address_augmented" in df.columns:
    aug = df[["noisy_address_augmented", "correct_address", "state"]].dropna(
        subset=["noisy_address_augmented"]
    ).rename(columns={"noisy_address_augmented": "input", "correct_address": "target"})
    pairs = pd.concat([pairs, aug], ignore_index=True)
    print(f"\nRows after including augmented pairs: {len(pairs):,}")

# ── Stratified split (paper: stratified sampling across states) ──────────────
train_df, temp_df = train_test_split(
    pairs, test_size=0.20, random_state=RANDOM_SEED, stratify=pairs["state"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_df["state"]
)

# Optional cap for quick testing (set MAX_TRAIN_SAMPLES=None to use all data)
if MAX_TRAIN_SAMPLES is not None and len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(MAX_TRAIN_SAMPLES, random_state=RANDOM_SEED)
    print(f"\n⚠️  Training capped at {MAX_TRAIN_SAMPLES:,} samples (MAX_TRAIN_SAMPLES is set)")

print(f"\n✅ Split complete:")
print(f"   Train      : {len(train_df):,}")
print(f"   Validation : {len(val_df):,}")
print(f"   Test       : {len(test_df):,}")

# Convert to HuggingFace Dataset objects
def to_hf_dataset(dataframe):
    return Dataset.from_dict({
        "input":  dataframe["input"].tolist(),
        "target": dataframe["target"].tolist(),
    })

dataset = DatasetDict({
    "train":      to_hf_dataset(train_df),
    "validation": to_hf_dataset(val_df),
    "test":       to_hf_dataset(test_df),
})
print("\n✅ HuggingFace DatasetDict created")
print(dataset)

In [ ]:
# ── Cell 6: Load Tokenizer & Model ───────────────────────────────────────────
# Loads a FRESH t5-small from HuggingFace (not the already fine-tuned one).
# This gives the model a clean slate to learn from your dataset.
print(f"Loading fresh {MODEL_NAME} from HuggingFace...")

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f"✅ Model loaded  |  Parameters: {model.num_parameters():,}")

In [ ]:
# ── Cell 7: Tokenise Datasets ─────────────────────────────────────────────────
# Prepends "rectify location: " to every noisy input (standard T5 task prefix).
# Labels are padded with -100 so the Trainer ignores padding tokens in the loss.

def tokenize(batch):
    inputs = [TASK_PREFIX + text for text in batch["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SEQ_LEN,
        padding=False,      # DataCollatorForSeq2Seq handles dynamic padding per batch
        truncation=True,
    )

    # text_target= is the correct API since transformers 4.15+
    # (as_target_tokenizer() context manager was removed and will crash on 4.40+)
    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_SEQ_LEN,
        padding=False,
        truncation=True,
    )

    # Replace pad token id with -100 → Trainer ignores these in cross-entropy loss
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels["input_ids"]
    ]
    return model_inputs

print("Tokenising datasets (this may take a few minutes on 500K rows)...")
tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["input", "target"],
    desc="Tokenising",
)
print("✅ Tokenisation complete")
print(tokenized)

In [ ]:
# ── Cell 8: Fuzzy Accuracy Metric ────────────────────────────────────────────
# Mirrors the exact logic used in accuracy.py (included in the blah/ folder).
# Paper reports accuracy using SequenceMatcher ratio ≥ 0.85 threshold.

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predicted token IDs to strings
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 (padding mask) before decoding labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    fuzzy_correct = 0
    exact_correct = 0

    for pred, label in zip(decoded_preds, decoded_labels):
        # Fuzzy match: SequenceMatcher ratio >= FUZZY_THRESHOLD counts as correct
        if SequenceMatcher(None, pred.lower(), label.lower()).ratio() >= FUZZY_THRESHOLD:
            fuzzy_correct += 1
        # Also track exact match (case-insensitive)
        if pred.lower() == label.lower():
            exact_correct += 1

    total = len(decoded_preds)
    return {
        "fuzzy_accuracy": round(fuzzy_correct / total * 100, 2),   # Main metric (paper)
        "exact_accuracy": round(exact_correct / total * 100, 2),   # Bonus metric
    }

print("✅ compute_metrics function defined")

## 🏋️ Train the Model

Sets up the HuggingFace `Seq2SeqTrainer` with all paper hyperparameters, then runs training.  
Early stopping will halt training if validation `fuzzy_accuracy` stops improving for 2 consecutive epochs.

In [ ]:
# ── Cell 9: Build Trainer & Train ─────────────────────────────────────────────

# Data collator: handles dynamic padding per batch (more memory efficient than padding all at once)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if USE_FP16 else None,   # Required for FP16 tensor cores
)

# Compute warmup steps from total training steps (paper: 5% of total steps)
steps_per_epoch = max(1, len(tokenized["train"]) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}")

# ── Seq2SeqTrainingArguments — exact paper configuration ─────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # ✏️ Epochs — also adjustable at the top of Cell 4
    num_train_epochs=NUM_EPOCHS,

    # ✏️ Batch size — also adjustable at the top of Cell 4
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,   # No gradients during eval → can use larger batch

    gradient_accumulation_steps=GRAD_ACCUM,      # Effective batch = BATCH_SIZE × GRAD_ACCUM

    # Optimizer: AdamW with weight decay (paper spec)
    optim="adamw_torch",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",                  # Linear warmup → linear decay
    warmup_steps=warmup_steps,

    # FP16 mixed precision (paper: Tesla P100 on Kaggle; auto-disabled on CPU)
    fp16=USE_FP16,

    # Evaluate at the end of every epoch to monitor val performance
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,                          # Keep only the 2 best checkpoints (saves disk)

    # Seq2seq: generate actual text predictions during eval (needed for our metric)
    predict_with_generate=True,
    generation_max_length=MAX_SEQ_LEN,

    # Load the best checkpoint at the end (used by EarlyStoppingCallback)
    load_best_model_at_end=True,
    metric_for_best_model="fuzzy_accuracy",
    greater_is_better=True,

    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=200,
    report_to="none",   # No wandb / tensorboard unless you add your key
    seed=RANDOM_SEED,
)

# ── Assemble Trainer ──────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        # Stop training early if val fuzzy_accuracy does not improve for 2 epochs
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

print("\n✅ Trainer ready. Starting training...")
print("=" * 60)
trainer.train()
print("=" * 60)
print("✅ Training complete!")

In [ ]:
# ── Cell 10: Evaluate on Held-Out Test Set ───────────────────────────────────
# Runs on the 10% test split that the model has never seen during training.
print("Running evaluation on test set...")

test_results = trainer.predict(tokenized["test"], metric_key_prefix="test")
metrics = test_results.metrics

print("\n" + "=" * 60)
print("📊 TEST SET RESULTS")
print("=" * 60)
print(f"  Fuzzy Accuracy  : {metrics.get('test_fuzzy_accuracy', 'N/A')}%")
print(f"  Exact Accuracy  : {metrics.get('test_exact_accuracy', 'N/A')}%")
print(f"  Test Loss       : {metrics.get('test_loss', 0):.4f}")
print("=" * 60)

# ── Show a few prediction samples ─────────────────────────────────────────────
print("\n🔍 Sample predictions from test set:")
sample_inputs  = test_df["input"].head(10).tolist()
sample_targets = test_df["target"].head(10).tolist()

model.eval()
for noisy, expected in zip(sample_inputs, sample_targets):
    input_ids = tokenizer(TASK_PREFIX + noisy, return_tensors="pt").input_ids.to(device)
    model.to(device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=MAX_SEQ_LEN)
    predicted = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    match = "✅" if SequenceMatcher(None, predicted.lower(), expected.lower()).ratio() >= FUZZY_THRESHOLD else "❌"
    print(f"  {match}  Input: {noisy:<40}  Expected: {expected:<25}  Got: {predicted}")

In [ ]:
# ── Cell 11: Save Model to Google Drive ──────────────────────────────────────
# Saves the best checkpoint in the same folder structure as t5_corrector_final/
# so it is a direct drop-in replacement in the project without changing any code.

print(f"Saving model to: {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n✅ Model saved! Files in output directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   {f:<40}  {size_mb:.1f} MB")

print(f"\n🎉 Done! Download {OUTPUT_DIR} and replace your local t5_corrector_final/ folder.")

In [ ]:
# ── Cell 12: Plot Training Curves ────────────────────────────────────────────
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

# Extract per-epoch validation metrics
val_loss, val_fuzzy, val_exact, epochs = [], [], [], []
for entry in log_history:
    if "eval_loss" in entry:
        val_loss.append(entry["eval_loss"])
        val_fuzzy.append(entry.get("eval_fuzzy_accuracy", 0))
        val_exact.append(entry.get("eval_exact_accuracy", 0))
        epochs.append(entry["epoch"])

# Extract training loss (logged every 200 steps)
train_steps, train_loss = [], []
for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_loss.append(entry["loss"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training loss
if train_steps:
    axes[0].plot(train_steps, train_loss, color="#e74c3c", linewidth=1.5)
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)

# Validation loss per epoch
if val_loss:
    axes[1].plot(epochs, val_loss, marker="o", color="#3498db", linewidth=2)
    axes[1].set_title("Validation Loss per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(alpha=0.3)

# Fuzzy and exact accuracy per epoch
if val_fuzzy:
    axes[2].plot(epochs, val_fuzzy, marker="o", label="Fuzzy Acc (≥0.85)", color="#2ecc71", linewidth=2)
    axes[2].plot(epochs, val_exact, marker="s", label="Exact Acc", color="#f39c12", linewidth=2, linestyle="--")
    axes[2].set_title("Validation Accuracy per Epoch")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Accuracy (%)")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

plt.suptitle("T5-Small Fine-Tuning — Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()

# Save the plot to Drive alongside the model
plot_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Plot saved to: {plot_path}")

## ✅ What to do after training

1. **Download the model folder** from Google Drive:  
   `My Drive/t5_location/t5_corrector_final/` → download as zip

2. **Replace the local model folder** in your project:  
   Extract and overwrite `Auto_correct_address/t5_corrector_final/`

3. **No other changes needed** — `T5_fine_tuned.py` and `integrated.py` already point to this folder.

---

### Files saved to Drive

| File | Description |
|---|---|
| `model.safetensors` | Fine-tuned model weights |
| `spiece.model` | SentencePiece tokenizer |
| `tokenizer_config.json` | Tokenizer settings |
| `config.json` | Model architecture config |
| `generation_config.json` | Generation settings |
| `training_curves.png` | Loss and accuracy plots |
| `logs/` | TensorBoard-compatible training logs |